<a id="mgs-17b-empirical"></a>
# MGS-17b — Selection empirique d'algorithmes : du podium a la carte probleme x representation x solveur

## Origine

Ce notebook distille le projet L4 EPITA SCIA 2026 de **Theodore Deguest** (projet solo) :

- PR source : [jsboigeEpita/2026-Epita-Intelligence-Symbolique#42](https://github.com/jsboigeEpita/2026-Epita-Intelligence-Symbolique/pull/42)
- Apport verifie : protocole commun applique a trois terrains structurants (Sudoku, Puissance 4, Wordle),
  avec metriques uniformes, timeouts, execution parallele, checkpoints CSV et visualisations.
- Reproduction WSL/Linux du 2026-08-24 : `uv run pytest -q` -> 20 tests passes en 21,11 s.
- Limite assumee : sous Windows natif, l'orchestrateur emploie les API Unix `resource` et `SIGALRM` ;
  la collecte echoue. Cette limite reste visible et motive la section Linux/WSL obligatoire.

Ce que ce notebook **n'est pas** : un copier-coller du depot etudiant. Le geste conserve est
**le protocole commun et la triangulation cross-terrains** ; le depot de Theodore contient les
details d'implementation et la documentation utilisateur (voir sa PR).

## Plan

| Section | Contenu | Statut |
|---|---|---|
| 1 (ce notebook) | Attribution, cadre theorique (Rice 1976, Smith-Miles 2009, NFL), setup env execute | **tranche 1/N** |
| 2 | Protocole commun : 3 terrains (Sudoku / Puissance 4 / Wordle), budgets, metriques homogenes | tranche 2/N |
| 3 | Pareto frontieres et cas discriminant (le gagnant depend de l'instance) | tranche 3/N |
| 4 | Ponts series : Search, Sudoku, GameTheory, App-7-Wordle, App-12-ConnectFour | tranche 4/N |

Les tranches 2 a 4 dependent du terrain de Theodore et arriveront dans des PR suivantes
de la lane `myia-po-2024:CoursIA-2`. Cette premiere tranche etablit le cadre et l'attribution.


## Cadre theorique : la selection empirique d'algorithmes

Trois pierres angulaires a poser avant d'attaquer la selection :

### 1. Le probleme de Rice (1976)

John R. Rice, *The Algorithm Selection Problem*, Advances in Computers vol. 15 (1976).
La formalisation : on se donne un ensemble $I$ d'instances, un ensemble $A$ d'algorithmes,
une fonction de performance $p : I \times A \to \mathbb{R}$ (a minimiser). On cherche une fonction
de selection $s : I \to A$ qui minimise $\sum_{i \in I} p(i, s(i))$.

La difficulte : $p$ n'est pas connue analytiquement. On observe $p$ sur un echantillon
d'instances et on apprend $s$. Trois ingredients :

- **instance features** : un vecteur $f(i) \in \mathbb{R}^k$ qui caracterise l'instance ;
- **algorithme portfolio** : l'ensemble $A$ des candidats ;
- **cout de selection** : le cout (en temps, en memoire ou en evaluations) pour choisir $s$.

La selection n'est utile que si $s$ est **beaucoup moins cher** que d'executer tous les algorithmes.
Si tous les $a \in A$ sont triviaux, selectionner ne sert a rien.

### 2. No Free Lunch (Wolpert 1996)

David H. Wolpert, *The Lack of A Priori Distinctions Between Learning Algorithms*,
Neural Computation vol. 8 no. 7 (1996). Le resultat central : sur l'**ensemble de toutes les
distributions possibles**, la performance moyenne de tout algorithme est la meme que celle
d'un randomiseur uniforme. Donc aucune superiority intrinseque d'un algorithme sur un autre.

Ce que cela dit vraiment : si l'on **restreint** la distribution (par exemple, les instances
de Sudoku, ou les grilles Wordle de 5 lettres), les differences emergent. La selection n'est
donc legitime que sur une distribution d'instances bien definie. Sortir de la distribution =
sortir du domaine ou la selection a ete calibree.

### 3. Empirical algorithmics (Smith-Miles 2009)

Kate A. Smith-Miles, *Cross-Disciplinary Perspectives on Meta-Learning for Algorithm Selection*,
ACM Computing Surveys vol. 41 no. 1 (2009). Le cadre met en lumiere trois pieges recurrents :

- **biais de representation** : mesurer uniquement sur des instances faciles ou uniquement sur
  des instances dures ;
- **biais de metrique** : confondre temps CPU, temps wallclock, nombre de noeuds explores,
  qualite de solution ; la Pareto frontiere expose la structure ;
- **biais de budget** : dire "mieux" sans fixer le budget ; avec un budget infini, beaucoup
  d'algorithmes triviaux deviennent gagnants.

Le geste empirique qui repond a ces pieges : **une seule table de sortie, plusieurs metriques,
memes budgets, meme horizon, meme grain de randomisation, meme representation du substrat**.

## Ce que la suite couvrira

Les sections 2 a 4 (tranches suivantes) appliqueront ce cadre au protocole de Theodore sur les
trois terrains. Elles montreront notamment :

- une meme instance de Sudoku traitee par 6 solveurs (DLX, CP-SAT, SMT, GA, MCTS, recuit simule)
  avec un meme budget et une meme metrique ;
- une meme position de Puissance 4 traitee par 4 solveurs avec alpha-beta, MCTS et negamax ;
- une meme grille Wordle traitee par 3 solveurs informationnels.

Le resultat attendu n'est pas un classement unique mais une **carte (probleme, representation,
algorithme) -> (temps, memoire, qualite)** ou chaque algorithme est gagnant dans une zone.


In [1]:
# Setup env : verifier la disponibilite des briques utilisees dans les tranches suivantes.
# Cette cellule execute sans dependre du depot etudiant de Theodore : elle valide simplement
# que l'ecosysteme Python du notebook (kernel `coursia-ml-training`) tient les invariants
# promis par les Sections A et B du depot source.

import sys
import platform
import importlib

print(f"Python : {sys.version}")
print(f"Plateforme : {platform.system()} {platform.release()} ({platform.machine()})")

# Inventaire des bibliotheques qui seront sollicitees dans les tranches 2 a 4.
expected_modules = {
    'numpy':   'tableaux numeriques',
    'pandas':  'tables de resultats / CSV',
    'matplotlib': 'visualisations',
    'scipy':   'solveurs numeriques (recuit simule, etc.)',
    'sklearn': 'selection algo / portfolio / features',
    'ortools': 'CP-SAT (Sudoku, Puissance 4)',
    'pysat':   'SAT/SMT (Sudoku encode SAT)',
    'z3':      'SMT (Sudoku encode SMT)',
}

present = {}
missing = []
for mod, role in expected_modules.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        present[mod] = ver
    except Exception as e:
        missing.append((mod, role, type(e).__name__))

print("\n--- Modules presents ---")
for mod, ver in present.items():
    print(f"  {mod:12s} {ver:>12s}  ({expected_modules[mod]})")
if missing:
    print("\n--- Modules manquants ---")
    for mod, role, exc in missing:
        print(f"  {mod:12s} INDISPONIBLE ({role}) -- {exc}")
else:
    print("\n--- Aucun module manquant ---")


Python : 3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:27:10) [MSC v.1944 64 bit (AMD64)]
Plateforme : Windows 10 (AMD64)



--- Modules presents ---
  numpy               2.4.6  (tableaux numeriques)
  pandas              3.0.3  (tables de resultats / CSV)
  matplotlib         3.11.0  (visualisations)
  scipy              1.17.1  (solveurs numeriques (recuit simule, etc.))
  sklearn             1.9.0  (selection algo / portfolio / features)
  ortools         9.15.6755  (CP-SAT (Sudoku, Puissance 4))
  pysat           1.9.dev15  (SAT/SMT (Sudoku encode SAT))
  z3                      ?  (SMT (Sudoku encode SMT))

--- Aucun module manquant ---


## Ponts series

Les tranches suivantes connecteront explicitement :

- **Sudoku** : la serie `MyIA.AI.Notebooks/Sudoku/` (notamment `Sudoku-18-Comparison-*.ipynb`)
  pour le terrain Sudoku (DLX, CP-SAT, SMT) ;
- **Search / Part1-Foundations** : Search-1 a Search-4 pour les bases de complexite ;
- **Search / Part2-CSP** : Search-5 a Search-12 pour les techniques CSP/SAT/SMT ;
- **Search / Applications/App-7-Wordle** : solveur Wordle informationnel ;
- **Search / Applications/App-12-ConnectFour** : solveur Puissance 4 (alpha-beta, MCTS) ;
- **GameTheory** : pour les sections en forme de jeu a deux joueurs (strategies mixtes, equilibrium).

## Sources

- **Source etudiante (protocole commun, implementation)** : Theodore Deguest, *Benchmark cross-paradigme de solveurs de jeux*, EPITA SCIA 2026, [PR #42](https://github.com/jsboigeEpita/2026-Epita-Intelligence-Symbolique/pull/42).
- **Rice 1976** : John R. Rice, *The Algorithm Selection Problem*, Advances in Computers 15, 1976.
- **Wolpert 1996** : David H. Wolpert, *The Lack of A Priori Distinctions Between Learning Algorithms*, Neural Computation 8(7), 1996.
- **Smith-Miles 2009** : Kate A. Smith-Miles, *Cross-Disciplinary Perspectives on Meta-Learning for Algorithm Selection*, ACM Computing Surveys 41(1), 2009.
